In [ ]:
pip install selenium

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Initialize the browser
driver = webdriver.Chrome()  # Use the path to your chromedriver if not in PATH
driver.get("https://iml.saltlakecounty.gov/IML")  # Replace with the actual URL

# Maximize the window (optional)
driver.maximize_window()

# Wait until the table is loaded
WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "body")))

In [ ]:
import time

inmate_data = []

search_button = driver.find_element(By.XPATH, "//a/img[@alt='Search Button']")
search_button.click()

def scrape_page():
    body = driver.find_element(By.XPATH, "/html/body/table[2]/tbody/tr[4]/td[3]/table/tbody/tr[3]/td/table/tbody/tr/td/table/tbody/tr[3]/td/table/tbody")
    rows = body.find_elements(By.TAG_NAME, "tr")
    for row in rows:
        columns = row.find_elements(By.TAG_NAME, "td")
        inmate_name = columns[0].text.strip()
        booking_number = columns[1].text.strip()
        inmate_data.append({
            'name': inmate_name,
            'booking_number': booking_number,
        })

def go_to_next_page():
    try:
        next_button = driver.find_element(By.LINK_TEXT, "Next>")
        next_button.click()
        time.sleep(2)  # Adjust sleep time as necessary
        return True
    except:
        return False  # No more pages

# Scrape all pages
counter = 1
while True and counter:
    scrape_page()
    print(f"scraped page: {counter}")
    counter = counter + 1
    if not go_to_next_page():
        break

print(f"Scraped {len(inmate_data)} inmates.")


In [ ]:
import csv
from datetime import datetime

# Get today's date
today = datetime.today()

# Format it as MM-DD-YYYY
formatted_date = today.strftime("%m-%d-%Y")

# Define the file name
csv_file = f"data/inmate_data-{formatted_date}.csv"

# Define the headers for the CSV file (adjust based on your data structure)
headers = ['name', 'booking_number']

# Write the data to a CSV file
with open(csv_file, mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=headers)
    
    # Write the header
    writer.writeheader()
    
    # Write the rows
    for inmate in inmate_data:
        writer.writerow(inmate)

print(f"Inmate data saved to {csv_file}")


In [ ]:
inmate_data2 = []
skipped = []
violent_crimes = ["MURDER", "MANSLAUGHTER", "AGGRAVATED ASSAULT", "ROBBERY", "KIDNAPPING", 
                  "RAPE"]


def search_inmate(inmate_name, booking_number):
    try:
        search_button = driver.find_element(By.XPATH, "/html/body/form[2]/table[2]/tbody/tr[4]/td[3]/table/tbody/tr[3]/td[5]/table/tbody/tr/td/table/tbody/tr[3]/td/table/tbody/tr/td/a[1]")
        search_input = driver.find_element(By.XPATH, "/html/body/form[2]/table[2]/tbody/tr[4]/td[3]/table/tbody/tr[3]/td[5]/table/tbody/tr/td/table/tbody/tr[2]/td/table/tbody/tr[2]/td[2]/input")
        search_input.clear()
        search_input.send_keys(booking_number)
        search_button.click()
        WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH, "/html/body/table[2]/tbody/tr[4]/td[3]/table/tbody/tr[3]/td/table/tbody/tr/td/table/tbody/tr[3]/td/table/tbody/tr")))


        row = driver.find_element(By.XPATH, "/html/body/table[2]/tbody/tr[4]/td[3]/table/tbody/tr[3]/td/table/tbody/tr/td/table/tbody/tr[3]/td/table/tbody/tr")
        row.click()
        WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH, "/html/body/table[2]/tbody/tr[4]/td[3]/table/tbody/tr[3]/td/table/tbody/tr/td/table/tbody/tr/td/table[2]/tbody/tr[3]/td[4]")))

        citizenship = driver.find_element(By.XPATH, "/html/body/table[2]/tbody/tr[4]/td[3]/table/tbody/tr[3]/td/table/tbody/tr/td/table/tbody/tr/td/table[2]/tbody/tr[3]/td[4]")
        citizenship_inner_html = citizenship.get_attribute("innerHTML")

        cob = driver.find_element(By.XPATH, "/html/body/table[2]/tbody/tr[4]/td[3]/table/tbody/tr[3]/td/table/tbody/tr/td/table/tbody/tr/td/table[2]/tbody/tr[4]/td[2]")
        cob_inner_html = cob.get_attribute("innerHTML")

        race = driver.find_element(By.XPATH, "/html/body/table[2]/tbody/tr[4]/td[3]/table/tbody/tr[3]/td/table/tbody/tr/td/table/tbody/tr/td/table[1]/tbody/tr[2]/td/div[2]/table/tbody/tr[3]/td[2]")
        race_inner_html = race.get_attribute("innerHTML")

        description_elements = driver.find_elements(By.XPATH, "//html/body/table[2]/tbody/tr[4]/td[3]/table/tbody/tr[3]/td/table/tbody/tr/td/table/tbody/tr/td/table[7]//td[@colspan='2']")
        descriptions = [element.text for element in description_elements]

        inmate_details = {
                'name': inmate_name,
                'booking_number': booking_number,
                'citizenship': citizenship_inner_html,
                'cob': cob_inner_html,
                'race': race_inner_html,
                'charges': "; ".join(descriptions)
            }
        inmate_data2.append(inmate_details)
        
        back_to_db_search = driver.find_element(By.XPATH, "/html/body/map[2]/area")
        back_to_db_search.click()
        print(f"Details for booking number {booking_number}: {inmate_details}")
        
        time.sleep(0.5)  # Adjust sleep time as necessary
    except Exception as e:
        print(f"Error for booking number {booking_number}: {str(e)}")
        inmate_details = {
                'name': inmate_name,
                'booking_number': booking_number,
            }
        skipped.append(inmate_details)
        driver.get("http://iml.slsheriff.org/IML")

driver.get("http://iml.slsheriff.org/IML")  # Replace with the actual URL

# Wait until the table is loaded
WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "body")))

# Example: Search for each inmate
for inmate in inmate_data:
    search_inmate(inmate['name'], inmate['booking_number'])


In [ ]:
for inmate in skipped:
    search_inmate(inmate['name'], inmate['booking_number'])

In [ ]:
# Define the file name
csv_file = "data/inmate_charges.csv"

# Define the headers for the CSV file (adjust based on your data structure)
headers = ['name', 'booking_number', 'citizenship', 'cob', 'race', 'charges']

# Write the data to a CSV file
with open(csv_file, mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=headers)
    
    # Write the header
    writer.writeheader()
    
    # Write the rows
    for inmate in inmate_data2:
        writer.writerow(inmate)

print(f"Inmate data saved to {csv_file}")


In [ ]:
# Define the file name
csv_file = "data/inmate_skipped.csv"

# Define the headers for the CSV file (adjust based on your data structure)
headers = ['name', 'booking_number']

# Write the data to a CSV file
with open(csv_file, mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=headers)
    
    # Write the header
    writer.writeheader()
    
    # Write the rows
    for inmate in skipped:
        writer.writerow(inmate)

print(f"Inmate data saved to {csv_file}")


In [48]:
driver.quit()